In [ ]:
import os
import glob
import pandas as pd

folder_path = r"***/umr_files"
custom_company = pd.read_csv("***/serpstat.txt", dtype=str)
medical_ids = pd.read_csv("***/UMR_IDs_medical.csv", dtype=str)
company_list = pd.read_csv("***/group_list.csv", dtype=str)

companies_without_ssn = [
    'serpstat', 
    'ringostat', 
    'inweb', 
    'academyocean', 
    'talanovyti'
]

In [ ]:
# Merge all the UMR files into one big file

umr_files = glob.glob(os.path.join(folder_path, '*.txt'))

list_of_dfs = []

for f in umr_files:
    df = pd.read_csv(f, sep="|", dtype=str)
    
    file_name = os.path.basename(f)

    df.insert(loc=0, column='file_name', value=file_name)
    
    list_of_dfs.append(df)

umr_union = pd.concat(list_of_dfs, ignore_index=True)

In [ ]:
# Check if all the files have sufficient amount of people

umr_union['file_name'].value_counts()

In [ ]:
# Extract file names from the merged UMR file to check if we have data from all the files we need to process

umr_union['file_name'] = (umr_union['file_name']
    .str.replace(r'\d+', '', regex=True)
    .str.replace('.txt', '', regex=False)
    .str.replace('_', '', regex=False)
)

file_names = umr_union['file_name'].drop_duplicates()

In [ ]:
# Function to merge IDs by SSN

def merge_by_ssn(df, medical_ids, companies_without_ssn):
    
    filtered = df[
        (~df['file_name'].isin(companies_without_ssn)) &
        (df['Social Security Number'].notna()) &
        (df['Medical Plan Member ID'].notna()) &
        (df['Relationship'] == 'Employee')
    ].copy()

    filtered['Social Security Number'] = (
        filtered['Social Security Number']
        .astype(str)
        .str.replace(r'\D', '', regex=True)
    )

    medical_ids['Social Security Number'] = (
        medical_ids['Social Security Number']
        .astype(str)
        .str.replace(r'\D', '', regex=True)
    )

    return pd.merge(filtered, medical_ids, on="Social Security Number", how='inner')[
        ['file_name', 'alias', 'email', 'Medical Plan Member ID', 'member_id']
    ]

In [ ]:
# Function to merge IDs by the key (Last Name + DOB)

def merge_by_key(df, medical_ids):
    
    medical_ids = medical_ids.copy()
    medical_ids['key'] = medical_ids['Last Name'].astype(str).str.lower() + medical_ids['DOB'].astype(str).str.lower()

    filtered = df[
        (df['Medical Plan Member ID'].notna()) &
        (df['Relationship'] == 'Employee')
    ].copy()

    filtered['key'] = filtered['Last Name'].astype(str).str.lower() + filtered['DOB'].astype(str).str.lower()

    return pd.merge(filtered, medical_ids, on="key", how='inner')[
        ['file_name', 'alias', 'email', 'Medical Plan Member ID', 'member_id']
    ]

In [ ]:
# Re-format custom group

custom_company.insert(loc=0, column='file_name', value='serpstat')

custom_company = custom_company.rename(columns={
    'INDIV_LAST_NM_TXT': 'Last Name',
    'INDIV_BIRTH_DT'   : 'DOB',
    'INDIV_SEQ_NBR'    : 'Relationship',
    'ID_CARD_VALUE'    : 'Medical Plan Member ID'
})

custom_company['DOB'] = pd.to_datetime(custom_company['DOB'], format='%Y%m%d').dt.strftime('%m/%d/%Y')

custom_company['Relationship'] = custom_company['Relationship'].replace('00', 'Employee')

custom_company = custom_company.apply(lambda x: x.str.strip() if x.dtype == 'object' else x)

custom_company = custom_company[[
    'file_name',
    'Last Name',
    'DOB',
    'Relationship',
    'Medical Plan Member ID'
]]

In [ ]:
# Merge IDs by SSN and key

merged_by_ssn = merge_by_ssn(umr_union, medical_ids, companies_without_ssn)

company_a = umr_union[umr_union['file_name'] == 'ringostat']
company_b = umr_union[umr_union['file_name'] == 'inweb']
company_c = umr_union[umr_union['file_name'] == 'academyocean']
company_d = umr_union[umr_union['file_name'] == 'talanovyti']
company_e = custom_group

ids_a = medical_ids[medical_ids['alias'] == 'ringostat']
ids_b = medical_ids[medical_ids['alias'] == 'inweb']
ids_c = medical_ids[medical_ids['alias'] == 'academyocean']
ids_d = medical_ids[medical_ids['alias'] == 'talanovyti']
ids_e = medical_ids[medical_ids['alias'] == 'serpstat']

merged_by_key = pd.concat([
    merge_by_key(company_a, ids_a),
    merge_by_key(company_b, ids_b),
    merge_by_key(company_c, ids_c),
    merge_by_key(company_d, ids_d),
    merge_by_key(company_e, ids_e)
]).reset_index(drop=True)

In [ ]:
# Export results

umr_union.to_csv("***/umr_union.csv", index=False)

export = pd.concat([merged_by_ssn, merged_by_key], ignore_index=True)
export.rename(columns={'Medical Plan Member ID': 'NEW medical_member_id', 'member_id': 'OLD medical_member_id'}, inplace=True)
export.to_csv("***/merged_medical_ids.csv", index=False)

file_names.to_csv("***/file_names.csv", index=False)

In [ ]:
# Validate groups that have zero new IDs

pd.set_option('display.max_rows', None)

export.loc[
    export['NEW medical_member_id'] != export['OLD medical_member_id'], 'alias'
].value_counts().reindex(company_list['alias'], fill_value=0).sort_values()